# Aviation Accident Safety — Exploratory Data Analysis & Recommendations

## Executive question

For professionally built airplanes involved in accidents from 1983 onward, which manufacturers and make/model combinations show comparatively low occupant serious/fatal injury rates and low aircraft-destruction rates? How do weather and phase of flight relate to these outcomes?

### Important interpretation constraint
This is **accident-record data**, not exposure data. We do not know how many safe flights each aircraft type completed. Therefore, the results describe **outcomes conditional on an accident appearing in this dataset**; they do not estimate the probability of having an accident in the first place.

## Step 1 — Load packages and cleaned data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter

os.makedirs("images", exist_ok=True)
pd.set_option("display.max_columns", 50)

# Explicit light plotting surfaces make figures readable in GitHub, Colab,
# and dark-mode notebook interfaces. Saved figures are never transparent.
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.transparent": False,
    "svg.fonttype": "none",
    "axes.edgecolor": "0.75",
    "grid.color": "0.88",
    "grid.linewidth": 0.8,
})

def finish_percent_axis(ax, axis="x"):
    """Format a 0-1 proportion axis as percentages and simplify chart borders."""
    if axis == "x":
        ax.xaxis.set_major_formatter(PercentFormatter(1.0))
        ax.grid(axis="x", alpha=0.65)
        ax.grid(axis="y", visible=False)
    else:
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
        ax.grid(axis="y", alpha=0.65)
        ax.grid(axis="x", visible=False)
    sns.despine(ax=ax)

def label_horizontal_bars(ax, values, samples=None):
    """Add percentage values (and optional sample size) to horizontal bars."""
    max_val = max(values) if len(values) else 0
    pad = max(max_val * 0.025, 0.004)
    for i, value in enumerate(values):
        suffix = f"  n={int(samples.iloc[i])}" if samples is not None else ""
        ax.text(value + pad, i, f"{value:.1%}{suffix}", va="center", fontsize=9)
    ax.set_xlim(0, max(max_val * 1.32, 0.05))

def save_figure(fig, filename):
    fig.savefig(
        f"images/{filename}",
        bbox_inches="tight",
        facecolor="white",
        edgecolor="none",
        transparent=False,
    )

df = pd.read_csv("data/AviationData_cleaned.csv", parse_dates=["Event.Date"])
print("Cleaned shape:", df.shape)
df.head()


## Analysis population and robustness rules

- Injury comparisons require `Total.Occupants > 0`.
- For each `Plane.Type`, we use its **mean observed occupants** as a size proxy. Types averaging **≤20 occupants** are classified as small; types averaging **>20** are classified as large. Classifying at the plane-type level avoids calling a Boeing 737 “small” simply because a particular event was a ferry or maintenance flight with few people aboard.
- Manufacturer comparisons require at least **20 injury-rate events** and at least **20 events with known damage** within a size group.
- Plane-type comparisons require at least **10 injury-rate events** and **10 known-damage events**, per the assignment.
- Factor comparisons shown below require at least **50 usable events** per category.

In [ ]:
analysis_df = df[df["Total.Occupants"] > 0].copy()

plane_type_stats = (
    analysis_df.groupby(["Make", "Plane.Type"])
    .agg(
        n=("Event.Id", "size"),
        mean_injury=("Serious.Fatal.Fraction", "mean"),
        median_injury=("Serious.Fatal.Fraction", "median"),
        destroyed_rate=("Destroyed", "mean"),
        damage_n=("Destroyed", "count"),
        mean_occupants=("Total.Occupants", "mean"),
        median_occupants=("Total.Occupants", "median"),
    )
    .reset_index()
)
plane_type_stats["Size.Group"] = np.where(
    plane_type_stats["mean_occupants"] <= 20, "Small", "Large"
)

size_map = plane_type_stats.set_index("Plane.Type")["Size.Group"]
analysis_df["Size.Group"] = analysis_df["Plane.Type"].map(size_map)

print(analysis_df["Size.Group"].value_counts())
plane_type_stats["Size.Group"].value_counts()

## Step 2A — Manufacturer injury-rate comparison

In [ ]:
make_stats = (
    analysis_df.groupby(["Size.Group", "Make"])
    .agg(
        n=("Event.Id", "size"),
        mean_injury=("Serious.Fatal.Fraction", "mean"),
        median_injury=("Serious.Fatal.Fraction", "median"),
        destroyed_rate=("Destroyed", "mean"),
        damage_n=("Destroyed", "count"),
        mean_occupants=("Total.Occupants", "mean"),
    )
    .reset_index()
)

eligible_makes = make_stats[(make_stats["n"] >= 20) & (make_stats["damage_n"] >= 20)].copy()
small_make_15 = eligible_makes[eligible_makes["Size.Group"] == "Small"].nsmallest(15, "mean_injury")
large_make_15 = eligible_makes[eligible_makes["Size.Group"] == "Large"].nsmallest(15, "mean_injury")

print("Small-aircraft manufacturers (lowest injury rates):")
display(small_make_15[["Make", "n", "mean_injury", "destroyed_rate"]])
print("Large-aircraft manufacturers (eligible):")
display(large_make_15[["Make", "n", "mean_injury", "destroyed_rate"]])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor="white")

small_plot = small_make_15.sort_values("mean_injury", ascending=False)
large_plot = large_make_15.sort_values("mean_injury", ascending=False)

for ax, plot_data, title in [
    (axes[0], small_plot, "Small aircraft"),
    (axes[1], large_plot, "Large aircraft"),
]:
    ax.barh(plot_data["Make"], plot_data["mean_injury"], alpha=0.9)
    label_horizontal_bars(ax, plot_data["mean_injury"].reset_index(drop=True),
                          plot_data["n"].reset_index(drop=True))
    finish_percent_axis(ax)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Mean serious/fatal injury fraction")
    ax.set_ylabel("")

fig.suptitle(
    "Manufacturer injury severity in observed accidents\n"
    "Eligible groups meet the analysis sample thresholds",
    fontsize=16, fontweight="bold"
)
fig.tight_layout(rect=[0, 0, 1, 0.92])
save_figure(fig, "make_injury_comparison.svg")
plt.show()


### Distribution of injury rates — small makes
A low mean can hide a long tail of severe accidents. The violin plot shows the full event-level distribution for the 10 eligible small-aircraft makes with the lowest mean injury fractions.

In [ ]:
small_top10_names = small_make_15.head(10)["Make"]
small_dist = analysis_df[
    (analysis_df["Size.Group"] == "Small") & analysis_df["Make"].isin(small_top10_names)
].copy()

fig, ax = plt.subplots(figsize=(14, 7), facecolor="white")
sns.violinplot(
    data=small_dist, x="Make", y="Serious.Fatal.Fraction",
    inner="quartile", cut=0, ax=ax
)
ax.tick_params(axis="x", rotation=45)
for label in ax.get_xticklabels():
    label.set_ha("right")
ax.set_xlabel("")
ax.set_ylabel("Serious/fatal injury fraction")
ax.set_title("Distribution of injury severity — 10 lowest-mean small-aircraft makes",
             fontweight="bold")
finish_percent_axis(ax, axis="y")
fig.tight_layout()
save_figure(fig, "small_make_injury_distribution.svg")
plt.show()


### Distribution of injury rates — large makes
Only five large-aircraft manufacturers satisfy the sample threshold, so all five are shown rather than implying that 10 robust groups exist.

In [ ]:
large_names = large_make_15["Make"]
large_dist = analysis_df[
    (analysis_df["Size.Group"] == "Large") & analysis_df["Make"].isin(large_names)
].copy()

fig, ax = plt.subplots(figsize=(12, 6), facecolor="white")
sns.stripplot(
    data=large_dist, x="Make", y="Serious.Fatal.Fraction",
    alpha=0.4, jitter=0.25, ax=ax
)
ax.tick_params(axis="x", rotation=30)
for label in ax.get_xticklabels():
    label.set_ha("right")
ax.set_xlabel("")
ax.set_ylabel("Serious/fatal injury fraction")
ax.set_title("Event-level injury severity — eligible large-aircraft makes",
             fontweight="bold")
finish_percent_axis(ax, axis="y")
fig.tight_layout()
save_figure(fig, "large_make_injury_distribution.svg")
plt.show()


## Step 2B — Aircraft destruction rate by manufacturer

In [ ]:
small_destroy_15 = eligible_makes[eligible_makes["Size.Group"] == "Small"].nsmallest(15, "destroyed_rate")
large_destroy_15 = eligible_makes[eligible_makes["Size.Group"] == "Large"].nsmallest(15, "destroyed_rate")

fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor="white")

small_plot = small_destroy_15.sort_values("destroyed_rate", ascending=False)
large_plot = large_destroy_15.sort_values("destroyed_rate", ascending=False)

for ax, plot_data, title in [
    (axes[0], small_plot, "Small aircraft"),
    (axes[1], large_plot, "Large aircraft"),
]:
    ax.barh(plot_data["Make"], plot_data["destroyed_rate"], alpha=0.9)
    label_horizontal_bars(
        ax,
        plot_data["destroyed_rate"].reset_index(drop=True),
        plot_data["damage_n"].reset_index(drop=True),
    )
    finish_percent_axis(ax)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Observed destroyed fraction")
    ax.set_ylabel("")

fig.suptitle(
    "Observed aircraft destruction fraction by manufacturer\n"
    "Labels show known-damage sample size",
    fontsize=16, fontweight="bold"
)
fig.tight_layout(rect=[0, 0, 1, 0.92])
save_figure(fig, "make_destruction_comparison.svg")
plt.show()

display(small_destroy_15[["Make", "n", "damage_n", "mean_injury", "destroyed_rate"]])
display(large_destroy_15[["Make", "n", "damage_n", "mean_injury", "destroyed_rate"]])


### Manufacturer findings

For small-aircraft records, **Maule** stands out as a balanced manufacturer-level candidate: 215 analyzed accidents, a mean serious/fatal injury fraction around 0.165, and an observed destruction rate around 4.2%. **Stinson**, **Diamond**, and **Aeronca** also combine below-cohort injury means with relatively low destruction rates. Manufacturer-level rankings should not be treated as causal because mission profile, fleet age, exposure, pilot characteristics, and reporting all differ by make.

For large-aircraft records, only five manufacturers clear the robustness threshold. **Bombardier** has the lowest observed destruction fraction (~6.0%) and a low injury fraction (~5.4%). **Boeing** has by far the largest large-aircraft sample (654 injury-rate events) and combines a mean injury fraction near 6.2% with a destruction fraction near 9.9%. McDonnell Douglas has the lowest mean injury fraction (~4.8%) but a higher destruction fraction (~14.9%), so it is not the strongest all-around recommendation.

## Step 2C — Specific airplane types

The client asked for model-level recommendations. We require at least 10 analyzed accidents and 10 known-damage observations for every plane type shown.

In [ ]:
eligible_types = plane_type_stats[
    (plane_type_stats["n"] >= 10) & (plane_type_stats["damage_n"] >= 10)
].copy()

small_types = eligible_types[eligible_types["Size.Group"] == "Small"].nsmallest(10, "mean_injury")
large_types = eligible_types[eligible_types["Size.Group"] == "Large"].sort_values("mean_injury")

print("Small types:")
display(small_types[["Plane.Type", "n", "damage_n", "mean_occupants", "mean_injury", "destroyed_rate"]])
print("Large types:")
display(large_types[["Plane.Type", "n", "damage_n", "mean_occupants", "mean_injury", "destroyed_rate"]])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 7), facecolor="white")

small_plot = small_types.sort_values("mean_injury", ascending=False)
large_plot = large_types.sort_values("mean_injury", ascending=False)

for ax, plot_data, title in [
    (axes[0], small_plot, "Small aircraft"),
    (axes[1], large_plot, "Large aircraft"),
]:
    ax.barh(plot_data["Plane.Type"], plot_data["mean_injury"], alpha=0.9)
    label_horizontal_bars(
        ax,
        plot_data["mean_injury"].reset_index(drop=True),
        plot_data["n"].reset_index(drop=True),
    )
    finish_percent_axis(ax)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Mean serious/fatal injury fraction")
    ax.set_ylabel("")

fig.suptitle(
    "Recommended plane types: observed serious/fatal injury fraction\n"
    "All shown types have ≥10 usable accident records; labels include n",
    fontsize=16, fontweight="bold"
)
fig.tight_layout(rect=[0, 0, 1, 0.90])
save_figure(fig, "model_recommendations.svg")
plt.show()


In [ ]:
# Distributional view for the same model groups
small_type_events = analysis_df[analysis_df["Plane.Type"].isin(small_types["Plane.Type"])]
large_type_events = analysis_df[analysis_df["Plane.Type"].isin(large_types["Plane.Type"])]

fig, axes = plt.subplots(1, 2, figsize=(17, 7), facecolor="white")
sns.stripplot(
    data=small_type_events, y="Plane.Type", x="Serious.Fatal.Fraction",
    alpha=0.38, jitter=0.25, ax=axes[0]
)
axes[0].set_title("Small types — event-level injury fractions", fontweight="bold")
axes[0].set_xlabel("Serious/fatal injury fraction")
axes[0].set_ylabel("")
finish_percent_axis(axes[0])

sns.stripplot(
    data=large_type_events, y="Plane.Type", x="Serious.Fatal.Fraction",
    alpha=0.38, jitter=0.25, ax=axes[1]
)
axes[1].set_title("Large types — event-level injury fractions", fontweight="bold")
axes[1].set_xlabel("Serious/fatal injury fraction")
axes[1].set_ylabel("")
finish_percent_axis(axes[1])

fig.tight_layout()
save_figure(fig, "model_injury_distributions.svg")
plt.show()


### Specific airplane recommendations

**Large passenger aircraft:**
- **Boeing 777** — 41 usable injury records; mean serious/fatal injury fraction ≈ **0.07%** and a destruction rate ≈ **4.8%** among 21 known-damage records. This is the strongest large-aircraft result in the eligible sample.
- **Bombardier CL-600-2B19** — 17 injury records; mean injury fraction ≈ **0.36%** and **0 observed destroyed aircraft** among 13 known-damage records. The sample is smaller, so this is a promising but less precise recommendation.
- **Boeing 757** — 23 injury records; mean injury fraction ≈ **4.5%** and 0 observed destroyed aircraft among 11 known-damage records.

**Small aircraft:**
- **Cessna 180J** — 28 injury records; mean serious/fatal fraction ≈ **3.6%**, with no destroyed aircraft among 27 known-damage records.
- **Piper PA-20** — 25 injury records; mean serious/fatal fraction ≈ **4.0%**, with no destroyed aircraft among 25 known-damage records.
- **Cessna 172SP**, **Diamond DA 20 C1**, and **Maule M-5-210C** each have 0 observed serious/fatal injury fraction and 0 observed destruction in their 11–12-event samples. These are encouraging, but the sample sizes are close to the minimum threshold and should be treated as less certain than the Cessna 180J or Piper PA-20 results.

These are conditional-on-accident outcomes, not fleet-wide accident-risk estimates.

## Step 3 — Other factor 1: Weather condition

We compare categories with at least 50 usable injury records. VMC means visual meteorological conditions and IMC means instrument meteorological conditions.

In [ ]:
def factor_summary(data, factor, min_n=50):
    out = (
        data.dropna(subset=[factor])
        .groupby(factor)
        .agg(
            n=("Event.Id", "size"),
            mean_injury=("Serious.Fatal.Fraction", "mean"),
            destroyed_rate=("Destroyed", "mean"),
            damage_n=("Destroyed", "count"),
        )
        .reset_index()
    )
    return out[(out["n"] >= min_n) & (out["damage_n"] >= min_n)].copy()

weather_stats = factor_summary(analysis_df, "Weather.Condition", min_n=50)
weather_stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), facecolor="white")

weather_plot = weather_stats.sort_values("mean_injury", ascending=False)
labels = [f"{row['Weather.Condition']} (n={int(row['n']):,})" for _, row in weather_plot.iterrows()]
axes[0].barh(labels, weather_plot["mean_injury"], alpha=0.9)
label_horizontal_bars(axes[0], weather_plot["mean_injury"].reset_index(drop=True))
finish_percent_axis(axes[0])
axes[0].set_title("Serious/fatal injury fraction", fontweight="bold")
axes[0].set_xlabel("Mean fraction")
axes[0].set_ylabel("")

weather_damage = weather_stats.sort_values("destroyed_rate", ascending=False)
damage_labels = [f"{row['Weather.Condition']} (n={int(row['damage_n']):,})" for _, row in weather_damage.iterrows()]
axes[1].barh(damage_labels, weather_damage["destroyed_rate"], alpha=0.9)
label_horizontal_bars(axes[1], weather_damage["destroyed_rate"].reset_index(drop=True))
finish_percent_axis(axes[1])
axes[1].set_title("Destroyed fraction", fontweight="bold")
axes[1].set_xlabel("Observed fraction")
axes[1].set_ylabel("")

fig.suptitle("Accident severity by weather condition", fontsize=16, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.90])
save_figure(fig, "weather_safety.svg")
plt.show()


### Weather finding

The difference is large and supported by substantial samples. In **VMC** there are roughly 14.6k usable events, with a mean serious/fatal injury fraction around **23%** and destruction around **7%**. In **IMC** there are roughly 923 events, with a mean serious/fatal injury fraction around **62%** and destruction around **37%**. Within accident records, IMC is therefore associated with much more severe outcomes. This is an association—not proof that weather alone causes the severity difference—because flight type, pilot training, aircraft type, terrain, and accident mechanism may differ between IMC and VMC events.

## Step 3 — Other factor 2: Broad phase of flight

Only phases with at least 50 injury records and 50 known-damage records are compared.

In [ ]:
phase_stats = factor_summary(analysis_df, "Broad.phase.of.flight", min_n=50)
phase_stats = phase_stats.sort_values("mean_injury")
phase_stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7.5), facecolor="white")

injury_plot = phase_stats.sort_values("mean_injury", ascending=False)
axes[0].barh(injury_plot["Broad.phase.of.flight"], injury_plot["mean_injury"], alpha=0.9)
label_horizontal_bars(axes[0], injury_plot["mean_injury"].reset_index(drop=True),
                      injury_plot["n"].reset_index(drop=True))
finish_percent_axis(axes[0])
axes[0].set_title("Serious/fatal injury fraction", fontweight="bold")
axes[0].set_xlabel("Mean fraction")
axes[0].set_ylabel("")

damage_plot = phase_stats.sort_values("destroyed_rate", ascending=False)
axes[1].barh(damage_plot["Broad.phase.of.flight"], damage_plot["destroyed_rate"], alpha=0.9)
label_horizontal_bars(axes[1], damage_plot["destroyed_rate"].reset_index(drop=True),
                      damage_plot["damage_n"].reset_index(drop=True))
finish_percent_axis(axes[1])
axes[1].set_title("Destroyed fraction", fontweight="bold")
axes[1].set_xlabel("Observed fraction")
axes[1].set_ylabel("")

fig.suptitle("Accident severity by phase of flight", fontsize=16, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.92])
save_figure(fig, "phase_safety.svg")
plt.show()


### Phase-of-flight finding

Severity varies strongly by phase. **Landing** accidents in the retained phase data have a mean serious/fatal injury fraction below **1%** and a destruction rate around **1.4%**, while **maneuvering** is roughly **36%** serious/fatal and **28%** destroyed; **climb** is roughly **32%** serious/fatal and **30%** destroyed. Takeoff and approach sit between these extremes. The phase field has substantial missingness, so this result should be interpreted as a pattern in records where phase is known, not as a complete census of all accidents.

## Final recommendations

### Large passenger aircraft
**Primary:** Boeing 777. It has the best observed injury outcome among large types that clear the 10-event threshold, with a larger injury sample than the other top-ranked models and a relatively low destruction rate.  
**Also consider:** Bombardier CL-600-2B19 and Boeing 757, with the explicit caveat that their known-damage samples are smaller.

### Small aircraft
**Primary:** Cessna 180J and Piper PA-20. Their samples are larger than the minimum and both combine low serious/fatal injury fractions with zero observed destruction among known-damage records.  
**Promising but smaller samples:** Cessna 172SP, Diamond DA 20 C1, Maule M-5-210C.

### Operational factors
1. **Weather matters:** IMC accidents are associated with sharply higher injury and destruction severity than VMC accidents.
2. **Phase of flight matters:** maneuvering and climb events are much more severe than landing/taxi events in the subset with known phase.

### Limitations
- The dataset contains accidents/incidents, not exposure hours or departures. A safe aircraft that flies millions of hours can appear often simply because it is widely used.
- Missing injury, damage, phase, and contextual fields can introduce selection bias.
- Manufacturer/model strings are manually normalized only where duplication is clear; residual naming variation may remain.
- Aircraft age, pilot experience, maintenance, geography, mission type, and regulatory regime are potential confounders.

The recommendations are therefore best used as **screening evidence for underwriting due diligence**, not as a standalone actuarial risk model.